# Notebook 03 - Train model du doan gia nha (Ha Noi)

Muc tieu:
- Train baseline + model chinh tren du lieu clean.
- Danh gia theo MAE/RMSE/R2.
- Luu model tot nhat vao `webDemo/BackEnd/model1.joblib`.

In [ ]:
%pip install numpy pandas scikit-learn joblib

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from joblib import dump

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "housing_hanoi_clean.csv"
MODEL_PATH = PROJECT_ROOT / "webDemo" / "BackEnd" / "model1.joblib"

DATA_PATH, MODEL_PATH

(WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha/data/processed/housing_hanoi_clean.csv'),
 WindowsPath('C:/Users/asus/Desktop/DuDoanGiaNha/webDemo/BackEnd/model1.joblib'))

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head(3)

## 1) Chon feature dung theo backend

In [ ]:
# Phai khop PredictorService.FEATURE_ORDER trong backend
FEATURES = [
    "chieuDai",
    "chieuNgang",
    "dienTich",
    "Phongngu",
    "SoTang",
    "PhongTam",
    "Loai",
    "GiayTo",
    "TinhTrangNoiThat",
    "Phuong",
]
TARGET = "Gia"

missing_features = [c for c in FEATURES if c not in df.columns]
if missing_features:
    raise ValueError(f"Thieu cot feature: {missing_features}")

X = df[FEATURES].copy()
y = pd.to_numeric(df[TARGET], errors="coerce")

valid_idx = y.notna()
X = X[valid_idx]
y = y[valid_idx]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
numeric_features = ["chieuDai", "chieuNgang", "dienTich", "Phongngu", "SoTang", "PhongTam"]
categorical_features = ["Loai", "GiayTo", "TinhTrangNoiThat", "Phuong"]

numeric_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 2) Train/Test split + train nhieu model

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    # CV chi de tham khao on dinh
    cv_neg_rmse = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=5,
        scoring="neg_root_mean_squared_error",
    )
    cv_rmse_mean = -cv_neg_rmse.mean()

    results.append({
        "model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "CV_RMSE": cv_rmse_mean,
    })
    fitted_pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values(by="RMSE", ascending=True)
results_df

NameError: name 'X' is not defined

## 3) Chon model tot nhat va luu vao backend

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_pipeline = fitted_pipelines[best_model_name]

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
dump(best_pipeline, MODEL_PATH)

print("Best model:", best_model_name)
print("Da luu model tai:", MODEL_PATH)

In [3]:
# Test nhanh 1 mau de dam bao model chay duoc
sample = X_test.head(1)
sample_pred = best_pipeline.predict(sample)[0]
print("Du doan mau test:", float(sample_pred))

NameError: name 'X_test' is not defined

## Mau bao cao (Notebook 03)

- Du lieu huan luyen: `data/processed/housing_hanoi_clean.csv`.
- Tap dac trung dau vao khop backend: `chieuDai`, `chieuNgang`, `dienTich`, `Phongngu`, `SoTang`, `PhongTam`, `Loai`, `GiayTo`, `TinhTrangNoiThat`, `Phuong`.
- Mo hinh duoc so sanh: Linear Regression (baseline), Random Forest, Gradient Boosting.
- Tieu chi danh gia: MAE, RMSE, R2 (kem CV_RMSE de danh gia on dinh).
- Mo hinh tot nhat duoc luu tai `webDemo/BackEnd/model1.joblib` de tich hop API `/predict`.